# Оценка параметров через ортогональные коллокации (OCFE)

Метод ортогональных коллокаций на конечных элементах (Orthogonal Collocation on Finite Elements) —
это предельный случай multiple shooting: вместо того чтобы интегрировать ОДУ внутри каждого шута
численным солвером, траектория внутри каждого элемента аппроксимируется полиномом, а требование
«полином удовлетворяет ОДУ в выбранных точках» становится **алгебраическим ограничением**.
Задача параметризации превращается в разреженную задачу нелинейных наименьших квадратов
с ограничениями-равенствами — ровно того вида, который уже решает `run_optimization_adaptive`
(см. `theory_gauss_newton.ipynb`).

Что это даёт по сравнению с текущей схемой (multiple shooting + интегрирование расширенной системы):

1. **Исчезает интегратор.** Невязки и все якобианы — это вычисления $f$, $f_x$, $f_\theta$, $h_x$, $h_\theta$
   в фиксированных узлах: батчевые вызовы CasADi `Function.map`, без последовательного шагания по времени.
2. **Не нужны вариационные уравнения.** Чувствительности $\partial x/\partial\theta$ не интегрируются —
   они неявно «решаются» разреженной линейной алгеброй шага Гаусса–Ньютона (см. раздел про якобианы).
3. **Жёсткие системы.** Коллокации по узлам Радо IIA — L-устойчивый неявный метод. Явные
   интеграторы (`dopri5` в JAX-пути, `RK45` в scipy-пути) на жёстких системах неприменимы.
4. **Входные сигналы** $u(t)$ вычисляются в узлах коллокации **один раз** до оптимизации —
   интерполяция данных уходит из итерационного цикла.

Постановка задачи та же, что и раньше:
$$
\dot{x} = f(x, \theta, u(t)), \qquad
y_i = h(x(t_i), \theta) + \varepsilon_i, \quad i = 1,\dots,N, \qquad
\varepsilon_i \sim \mathcal{N}(0, \Sigma_i),
$$
$$
\min_{\theta,\, x(\cdot)} \; \frac12 \sum_{i=1}^{N} \| y_i - h(x(t_i), \theta) \|^2_{W_i}
\quad \text{при условии} \quad \dot{x} = f(x, \theta, u(t)).
$$


## Аппроксимация траектории на конечных элементах

Разобьём $[t_0, t_f]$ на $N_e$ элементов сеткой $t_0 = s_0 < s_1 < \dots < s_{N_e} = t_f$,
$h_e = s_e - s_{e-1}$. Внутри элемента $e$ вводим локальное время $\tau \in [0, 1]$:
$t = s_{e-1} + \tau h_e$.

Состояние на элементе аппроксимируется полиномом Лагранжа **степени $K$**
по $K{+}1$ точкам $\{\tau_0 = 0,\ \tau_1, \dots, \tau_K\}$:

$$
x^e(\tau) = \sum_{j=0}^{K} \ell_j(\tau)\, X^e_j,
\qquad
\ell_j(\tau) = \prod_{m \ne j} \frac{\tau - \tau_m}{\tau_j - \tau_m},
$$

где $X^e_j \in \mathbb{R}^{n_x}$ — значение состояния в узле $\tau_j$ элемента $e$.
Точка $\tau_0 = 0$ — только интерполяционная (левый край элемента),
а $\tau_1 < \dots < \tau_K$ — **коллокационные узлы Радо IIA** (корни полинома Радо):

| $K$ | узлы $\tau_1,\dots,\tau_K$ | порядок $2K{-}1$ |
|---|---|---|
| 1 | $1$ (неявный Эйлер) | 1 |
| 2 | $1/3,\; 1$ | 3 |
| 3 | $\frac{4-\sqrt6}{10} \approx 0.155051,\;\; \frac{4+\sqrt6}{10} \approx 0.644949,\;\; 1$ | 5 |

Ключевые свойства Радо IIA:

- **правый узел $\tau_K = 1$** совпадает с концом элемента — значение $x^e(1) = X^e_K$ хранится явно,
  и непрерывность между элементами обеспечивается **общей переменной**:
  $X^{e+1}_0 \equiv X^e_K$ (никаких экстраполяций и отдельных ограничений склейки);
- метод **A- и L-устойчив** — пригоден для жёстких систем;
- порядок точности в концах элементов $2K-1$ (суперсходимость), внутри узлов — $K$.

> **Замечание к прототипу `experiments/collocation/test.ipynb`.** Там полином строится степени $K{-}1$
> только по коллокационным точкам $\{\tau_1,\dots,\tau_K\}$, а левый край элемента восстанавливается
> экстраполяцией $\ell_j(0)$. Такая постановка некорректна: матрица дифференцирования $D_{kj} = \ell_j'(\tau_k)$
> размера $K \times K$ вырождена (константы лежат в ядре, $\sum_j D_{kj} = 0$, ранг $K{-}1$), и система
> «$K$ коллокационных уравнений + непрерывность» переопределена на $n_x$ уравнений на элемент.
> Стандартная постановка со степенью $K$ и точкой $\tau_0 = 0$ (ниже) этой проблемы не имеет —
> численная проверка порядка в конце ноутбука.


## Коллокационные уравнения

Требуем, чтобы производная полинома совпадала с правой частью ОДУ в коллокационных узлах.
Обозначим $\tilde D_{kj} = \ell_j'(\tau_k)$ — матрицу размера $K \times (K{+}1)$, $k = 1..K$, $j = 0..K$
(теперь её квадратный блок по $j = 1..K$ невырожден). Тогда для каждого элемента $e$ и узла $k$:

$$
g^e_k \;=\; \sum_{j=0}^{K} \tilde D_{kj}\, X^e_j \;-\; h_e\, f\!\bigl(X^e_k,\ \theta,\ u(t^e_k)\bigr) \;=\; 0,
\qquad t^e_k = s_{e-1} + \tau_k h_e .
$$

Это $N_e \cdot K \cdot n_x$ алгебраических уравнений. Вместе с начальным условием
$X^1_0 = c_0$ (если $c_0$ известно; если оцениваем — $X^1_0$ просто остаётся свободной переменной)
они полностью заменяют интегрирование ОДУ.

**Вектор неизвестных** (в том же порядке, что `theta_full` в коде — сначала $\theta$, потом состояния):

$$
p = \bigl[\theta;\; X^1_0;\; X^1_1, \dots, X^1_K;\; X^2_1, \dots, X^2_K;\; \dots;\; X^{N_e}_K\bigr]
\in \mathbb{R}^{\,n_\theta + n_x (N_e K + 1)}.
$$

Здесь $X^{e+1}_0$ не хранится отдельно — это та же переменная, что $X^e_K$
(непрерывность «зашита» в структуру, как переиспользование $c_{j+1}$ в multiple shooting,
но точно, а не через штраф).

**Измерения.** Момент $t_i$ попадает в элемент $e_i$ с локальным временем
$\tau^{(i)} = (t_i - s_{e_i-1})/h_{e_i}$. Предсказание состояния — **линейная** комбинация узловых значений:

$$
x(t_i) = \sum_{j=0}^{K} \ell_j\bigl(\tau^{(i)}\bigr)\, X^{e_i}_j,
\qquad
r_i = y_i - h\bigl(x(t_i), \theta\bigr).
$$

Если сетку элементов привязать к сетке измерений (например, элемент на каждые $m$ интервалов
между измерениями), то для измерений на границах элементов $x(t_i) = X^{e}_K$ —
интерполяция вообще не нужна.


## Якобианы: всё алгебраика

В multiple shooting для якобианов приходится интегрировать вариационные уравнения
$\dot J_\theta = f_x J_\theta + f_\theta$, $\dot J_c = f_x J_c$ вместе с состоянием
(расширенная система размерности $n_x(1 + n_\theta + n_x)$). В коллокациях **ничего интегрировать
не нужно** — все производные выписываются явно.

**Якобиан невязок измерений** ($H$ в обозначениях `theory_gauss_newton.ipynb`).
Поскольку $x(t_i)$ линеен по узловым состояниям:

$$
\frac{\partial r_i}{\partial X^{e_i}_j} = -\,h_x\bigl(x(t_i), \theta\bigr)\, \ell_j\bigl(\tau^{(i)}\bigr),
\qquad
\frac{\partial r_i}{\partial \theta} = -\,h_\theta\bigl(x(t_i), \theta\bigr).
$$

**Якобиан коллокационных ограничений** ($G$). Дифференцируем $g^e_k$:

$$
\frac{\partial g^e_k}{\partial X^e_j} = \tilde D_{kj}\, I_{n_x} \;-\; h_e\, \delta_{kj}\, f_x\bigl(X^e_k, \theta\bigr),
\qquad
\frac{\partial g^e_k}{\partial \theta} = -\,h_e\, f_\theta\bigl(X^e_k, \theta\bigr).
$$

Столбец $j = 0$ элемента $e{+}1$ попадает в переменную $X^e_K$ — отсюда **блочно-бидиагональная**
структура $G$ по состояниям (в точности как блоки $J_{c}^{prev}$ и $-I$ в текущем `_solve_batch`,
только вместо матрицы перехода шута стоит $\tilde D_{k0} I$).

Все входящие величины — $f_x$, $f_\theta$, $h_x$, $h_\theta$ в узлах — вычисляются **батчево** одним
вызовом CasADi `Function.map` на все узлы сразу (тот же приём, что `observation_batch`).
Входы $u(t^e_k)$ — константы, предвычисленные до оптимизации.

### Где «спрятались» чувствительности

Линеаризуем коллокационный блок элемента $e$ (обозначим $F_x^{ek} = f_x(X^e_k,\theta)$, $F_\theta^{ek} = f_\theta(X^e_k,\theta)$):

$$
\underbrace{\Bigl(\tilde D_{\cdot,1:K} \otimes I_{n_x} - h_e \operatorname{blkdiag}(F_x^{e1},\dots,F_x^{eK})\Bigr)}_{M_e}
\begin{bmatrix}\delta X^e_1\\ \vdots\\ \delta X^e_K\end{bmatrix}
= -\,\tilde D_{\cdot,0} \otimes I_{n_x}\, \delta X^e_0 \;+\; h_e \begin{bmatrix}F_\theta^{e1}\\ \vdots\\ F_\theta^{eK}\end{bmatrix} \delta\theta .
$$

Решив её относительно $\delta X^e_K$, получаем
$\delta X^e_K = A_e\, \delta X^e_0 + B_e\, \delta\theta$ — это ровно дискретные матрицы
чувствительности $J_c(\tau_e)$ и $J_\theta(\tau_e)$, которые multiple shooting получает
интегрированием вариационных уравнений. Разница в том, что здесь их **никто явно не вычисляет**:
исключение $\delta X$ происходит внутри разреженного LU-разложения системы ККТ.
Это и есть «алгебраические уравнения для якобианов, решаемые численно» —
идея реализуется не отдельным алгоритмом, а самой структурой задачи.


## Шаг Гаусса–Ньютона и связь с кодом

Линеаризованная подзадача имеет тот же вид, что в multiple shooting:

$$
\min_{\delta p} \; \frac12 \| H\, \delta p - r \|_W^2
\quad \text{s.t.} \quad G\, \delta p = -g(p),
$$

и решается той же регуляризованной системой ККТ (метод квадратичного штрафа, $\mu = 1/\rho$):

$$
\begin{bmatrix}
H^T W H + \operatorname{reg} & G^T \\
G & -\mu I
\end{bmatrix}
\begin{bmatrix}
\delta p \\ \lambda
\end{bmatrix}
=
\begin{bmatrix}
H^T W r \\ -g
\end{bmatrix}.
$$

Отличия от multiple shooting чисто количественные:

| | Multiple shooting | Коллокации |
|---|---|---|
| Неизвестные | $\theta$ + $T$ начальных состояний ($n_\theta + T n_x$) | $\theta$ + все узловые состояния ($n_\theta + (N_e K + 1) n_x$) |
| Ограничения $G$ | непрерывность шутов, $(T{-}1) n_x$ строк | коллокационные уравнения, $N_e K n_x$ строк |
| Сборка $H$, $G$ | интегрирование расширенной системы на каждом шуте | батчевые вычисления $f_x, f_\theta, h_x, h_\theta$ в узлах |
| Стоимость итерации | $\sim$ ODE-солвер (90% времени) | $\sim$ одно разреженное LU блочно-трёхдиагональной матрицы |
| Пробный шаг (`trial_cost`) | полное интегрирование | подстановка в $r$ и $g$ (алгебраика) |

Матрица ККТ разреженная со структурой «стрелки»: плотные столбцы $\theta$
плюс блочно-трёхдиагональная часть по узловым состояниям. `scipy.sparse.linalg.splu`
решает такие системы за время, линейное по числу элементов.

**Переиспользование кода:** `run_optimization_adaptive`, `gn_step` и
`NormalEquations.covariance_theta` работают с интерфейсом $(J, R, J_G, R_G)$ и не знают,
откуда взялись матрицы. Достаточно класса `CollocationProblem` с методом
`solve(theta_full) -> (J, R, J_G, R_G)` и аналогом `make_full_theta`
(инициализация узловых состояний из измерений через `inverse_h`) — весь цикл оптимизации,
принятие/отклонение шагов, $\mu$-продолжение и доверительные интервалы остаются как есть.


## Ковариация параметров

Полностью повторяет `theory_gauss_newton.ipynb`: складываем измерительные и коллокационные
невязки в общую систему

$$
J_{\text{full}} = \begin{bmatrix} \sqrt{W}\, H \\ G \end{bmatrix}, \qquad
\operatorname{Cov}(\hat p) = \hat\sigma^2 \bigl( J_{\text{full}}^T J_{\text{full}} \bigr)^{-1},
\qquad
\hat\sigma^2 = \frac{\|R_{\text{full}}\|^2}{n_{\text{rows}} - \dim p},
$$

и берём **маргинальную ковариацию $\theta$** как $\theta$-блок обратной матрицы
(Шур-комплемент по блоку узловых состояний):

$$
\operatorname{Cov}(\hat\theta) = \hat\sigma^2 \bigl( A_{\theta\theta} - A_{\theta X} A_{XX}^{-1} A_{X\theta} \bigr)^{-1}.
$$

Число степеней свободы: $\text{dof} = n_y N + N_e K n_x - \dim p$. Обратите внимание,
что узловых состояний много, но и коллокационных «невязок» столько же — dof определяется,
как и раньше, в основном числом измерений.


## Жёсткие системы и выбор сетки

**Почему коллокации обязательны для жёстких систем.** Явные методы (текущие `dopri5` и `RK45`)
имеют ограниченную область устойчивости: на жёсткой системе с быстрой константой времени
$\lambda_{fast}$ шаг обязан быть $\sim 1/|\lambda_{fast}|$ даже там, где решение гладкое, —
интегратор «умирает», а не замедляется. Радо IIA **L-устойчив**: жёсткие моды подавляются при
любом $h_e$, и шаг выбирается по точности, а не по устойчивости. Численная демонстрация — ниже
($\lambda = -10^6$, один элемент).

**Практические правила выбора сетки:**

- узлов на элемент: $K = 3$ (порядок 5) — стандартный выбор; $K = 2$, если данных много и точность
  ограничена шумом;
- сетку элементов привязывать к сетке измерений: один элемент на $m = 1{-}4$ интервалов между
  измерениями (для данных 100 Гц и автомобильной динамики $m = 2{-}4$ достаточно);
- контроль достаточности сетки: решить с $N_e$ и $2 N_e$ элементов — оценки $\hat\theta$ должны
  совпасть в пределах доверительных интервалов;
- при сильно разномасштабных состояниях — обезразмеривание (масштабирование строк $g^e_k$ и
  переменных $X$), иначе LU по ККТ теряет точность.


## Альтернатива: коллокации как неявный интегратор (рекурсивное накопление якобианов)

Полная дискретизация (выше) делает узловые состояния неизвестными оптимизации. Есть второй
способ использовать те же коллокационные уравнения: как **одношаговый неявный интегратор**
внутри shooting. Тогда число неизвестных остаётся прежним ($\theta$ + начальное состояние
на каждом shooting-интервале), а якобианы **накапливаются рекурсивно** — без интегрирования
вариационных уравнений и без хранения большой матрицы $J$.

### Шаг 1. Отделяем левый край элемента

Коллокационное уравнение элемента $e$ в узле $k$ (из раздела выше):

$$
\sum_{j=0}^{K} \tilde D_{kj}\, X^e_j \;=\; h_e\, f\!\bigl(X^e_k,\ \theta,\ u^e_k\bigr),
\qquad k = 1, \dots, K .
$$

В сумме участвует $X^e_0$ — состояние на **левом краю** элемента, которое по непрерывности
равно состоянию в конце предыдущего элемента: $X^e_0 = x_{e-1}$. Вынесем его отдельно.
Разобьём матрицу $\tilde D \in \mathbb{R}^{K \times (K+1)}$ на первый столбец и остальное:

$$
\tilde D = [\, d_0 \mid D_1 \,],
\qquad d_0 \in \mathbb{R}^{K} \ (\text{столбец } j{=}0),
\qquad D_1 \in \mathbb{R}^{K \times K} \ (\text{столбцы } j{=}1..K).
$$

$D_1$ невырождена (проверено численно в код-ячейке выше: ранг $K$). Уравнение узла $k$
принимает вид

$$
d_{0,k}\, x_{e-1} \;+\; \sum_{j=1}^{K} (D_1)_{kj}\, X^e_j \;=\; h_e\, f(X^e_k, \theta, u^e_k).
$$

### Шаг 2. Стековая (векторная) форма

Соберём все стадии элемента в один вектор (это и есть $z$):

$$
z_e = \begin{bmatrix} X^e_1 \\ X^e_2 \\ \vdots \\ X^e_K \end{bmatrix} \in \mathbb{R}^{K n_x},
\qquad
F(z_e, \theta) = \begin{bmatrix} f(X^e_1, \theta, u^e_1) \\ f(X^e_2, \theta, u^e_2) \\ \vdots \\ f(X^e_K, \theta, u^e_K) \end{bmatrix} \in \mathbb{R}^{K n_x}.
$$

Все $K$ уравнений сразу записываются через кронекеровы произведения
(каждый скалярный коэффициент $(D_1)_{kj}$ действует на целый блок размера $n_x$):

$$
(d_0 \otimes I_{n_x})\, x_{e-1} \;+\; (D_1 \otimes I_{n_x})\, z_e \;=\; h_e\, F(z_e, \theta).
$$

Размерности: $d_0 \otimes I \in \mathbb{R}^{K n_x \times n_x}$,
$D_1 \otimes I \in \mathbb{R}^{K n_x \times K n_x}$.

### Шаг 3. Разрешаем относительно $z_e$

Умножаем слева на $(D_1 \otimes I)^{-1} = D_1^{-1} \otimes I$:

$$
z_e \;=\; \underbrace{-\bigl(D_1^{-1} d_0\bigr) \otimes I_{n_x}}_{A}\; x_{e-1}
\;+\; h_e\, \underbrace{\bigl(D_1^{-1} \otimes I_{n_x}\bigr)}_{B}\, F(z_e, \theta),
$$

то есть искомая одношаговая неявная схема:

$$
\boxed{\; z_e = A\, x_{e-1} + h_e\, B\, F(z_e, \theta), \qquad
A = -(D_1^{-1} d_0) \otimes I, \quad B = D_1^{-1} \otimes I. \;}
$$

### Шаг 4. Матрица $A$ — это просто «копирование» $x_{e-1}$ в каждую стадию

Интерполяционный полином точен на константах: если все $X^e_j$ равны одному и тому же
вектору, его производная равна нулю. В терминах $\tilde D$ это значит, что сумма каждой
строки равна нулю:

$$
\sum_{j=0}^{K} \tilde D_{kj} = 0
\quad\Longleftrightarrow\quad
d_0 + D_1 \mathbf{1}_K = 0
\quad\Longleftrightarrow\quad
d_0 = -D_1 \mathbf{1}_K,
$$

где $\mathbf{1}_K = [1, \dots, 1]^T$. Подставляя в определение $A$:

$$
A = -(D_1^{-1} d_0) \otimes I = (D_1^{-1} D_1 \mathbf{1}_K) \otimes I = \mathbf{1}_K \otimes I_{n_x}
= \begin{bmatrix} I_{n_x} \\ I_{n_x} \\ \vdots \\ I_{n_x} \end{bmatrix}.
$$

То есть $A x_{e-1}$ — это вектор $x_{e-1}$, скопированный в каждую из $K$ стадий.
Покомпонентно схема приобретает классический вид **стадийных уравнений неявного
метода Рунге–Кутты**:

$$
X^e_k = x_{e-1} + h_e \sum_{j=1}^{K} a_{kj}\, f(X^e_j, \theta, u^e_j),
\qquad a = D_1^{-1} \ \text{— таблица Бутчера}.
$$

Для узлов Радо $a = D_1^{-1}$ — это в точности табличные коэффициенты метода **Радо IIA**
(численная проверка в ячейке ниже). Значение в конце элемента извлекается последним блоком:

$$
x_e \;=\; (e_K^T \otimes I_{n_x})\, z_e \;=\; X^e_K,
\qquad e_K = [0, \dots, 0, 1]^T \in \mathbb{R}^K,
$$

— именно потому, что у Радо IIA правый узел $\tau_K = 1$ совпадает с концом элемента.
(Например, для узлов Гаусса–Лежандра это не так, и понадобилась бы отдельная интерполяция.)

### Шаг 5. Решение уравнения стадий: метод Ньютона

На каждом элементе решается нелинейное алгебраическое уравнение относительно $z$.
Резидуал и его якобиан:

$$
\Phi(z) = z - A x_{e-1} - h_e B F(z, \theta),
\qquad
\frac{\partial \Phi}{\partial z} = I - h_e B\, J_F(z),
$$

где

$$
J_F(z) = \frac{\partial F}{\partial z} =
\operatorname{blkdiag}\bigl(f_x(X^e_1), \dots, f_x(X^e_K)\bigr) \in \mathbb{R}^{K n_x \times K n_x}
$$

— блочно-диагональная матрица из якобианов правой части в стадиях
(стадия $k$ зависит только от $X^e_k$). Итерация Ньютона:

$$
M \,\Delta z = -\Phi(z), \qquad M = I - h_e B J_F(z), \qquad z \leftarrow z + \Delta z .
$$

Практические детали:
- стартовое приближение — $z = A x_{e-1}$ (константа) или экстраполяция полинома
  предыдущего элемента; обычно хватает 2–4 итераций;
- $M$ имеет размер $K n_x \times K n_x$ (при $K{=}3$, $n_x{=}2$ — всего $6 \times 6$):
  одно LU-разложение на итерацию стоит копейки;
- допустимо «замораживать» $M$ (упрощённый Ньютон): пересобирать $J_F$ не на каждой итерации;
- схема **неявная**, поэтому L-устойчивость сохраняется — жёсткость системы не ограничивает $h_e$.


### Шаг 6. Рекурсии для якобианов (вывод через неявное дифференцирование)

Нужны чувствительности решения в конце интервала по неизвестным задачи: по параметрам
$\theta$ и по начальному состоянию интервала $c$ (у single shooting $c = c_0$). Обозначим

$$
S^\theta_e = \frac{\partial x_e}{\partial \theta} \in \mathbb{R}^{n_x \times n_\theta},
\qquad
S^c_e = \frac{\partial x_e}{\partial c} \in \mathbb{R}^{n_x \times n_x}.
$$

Сошедшееся решение стадий $z_e$ — неявная функция от $x_{e-1}$ и $\theta$, заданная
уравнением $z_e = A x_{e-1} + h_e B F(z_e, \theta)$. Дифференцируем **обе части** по $\theta$,
аккуратно применяя цепное правило к $F(z_e(\theta), \theta)$ — у него два вхождения $\theta$,
явное и через $z_e$:

$$
\frac{d z_e}{d \theta}
= A \underbrace{\frac{d x_{e-1}}{d \theta}}_{S^\theta_{e-1}}
+ h_e B \left( \underbrace{\frac{\partial F}{\partial z}}_{J_F} \frac{d z_e}{d \theta}
+ \underbrace{\frac{\partial F}{\partial \theta}}_{F_\theta} \right),
\qquad
F_\theta = \begin{bmatrix} f_\theta(X^e_1) \\ \vdots \\ f_\theta(X^e_K) \end{bmatrix}
\in \mathbb{R}^{K n_x \times n_\theta}.
$$

Переносим член с $d z_e / d\theta$ влево и группируем:

$$
\bigl(\, I - h_e B J_F \,\bigr) \frac{d z_e}{d \theta}
= A\, S^\theta_{e-1} + h_e B F_\theta .
$$

Слева стоит **та же матрица $M$, что и в методе Ньютона** (шаг 5), вычисленная в сошедшейся
точке. Аналогично по $c$ (здесь явной зависимости $F$ от $c$ нет, остаётся только цепочка
через $x_{e-1}$):

$$
M\, \frac{d z_e}{d c} = A\, S^c_{e-1}.
$$

Итого — чувствительности всех стадий:

$$
\frac{d z_e}{d \theta} = M^{-1} \bigl( A S^\theta_{e-1} + h_e B F_\theta \bigr),
\qquad
\frac{d z_e}{d c} = M^{-1} A\, S^c_{e-1},
$$

а чувствительности конца элемента получаются извлечением последнего блока
($x_e = (e_K^T \otimes I) z_e$):

$$
S^\theta_e = (e_K^T \otimes I)\, M^{-1} \bigl( A S^\theta_{e-1} + h_e B F_\theta \bigr),
\qquad
S^c_e = (e_K^T \otimes I)\, M^{-1} A\, S^c_{e-1}.
$$

### Шаг 7. Матрицы перехода элемента и итоговые рекурсии

Введём два объекта, зависящие только от текущего элемента (вычисляются при сошедшемся $z_e$):

$$
\Psi_e = (e_K^T \otimes I)\, M^{-1} A \in \mathbb{R}^{n_x \times n_x}
\qquad \text{— матрица перехода состояния,}
$$
$$
\Gamma_e = (e_K^T \otimes I)\, M^{-1}\, h_e B F_\theta \in \mathbb{R}^{n_x \times n_\theta}
\qquad \text{— вклад параметров на элементе.}
$$

Тогда рекурсии, выражающие якобианы через себя на предыдущем шаге:

$$
\boxed{\;
S^c_e = \Psi_e\, S^c_{e-1}, \qquad
S^\theta_e = \Psi_e\, S^\theta_{e-1} + \Gamma_e,
\qquad S^c_0 = I, \quad S^\theta_0 = 0 .
\;}
$$

Развернём несколько шагов, чтобы «накопление» стало видно явно:

$$
S^c_1 = \Psi_1, \qquad
S^c_2 = \Psi_2 \Psi_1, \qquad
S^c_e = \Psi_e \Psi_{e-1} \cdots \Psi_1 ,
$$
$$
S^\theta_1 = \Gamma_1, \qquad
S^\theta_2 = \Psi_2 \Gamma_1 + \Gamma_2, \qquad
S^\theta_e = \sum_{m=1}^{e} \bigl( \Psi_e \cdots \Psi_{m+1} \bigr)\, \Gamma_m .
$$

Каждый элемент вносит свой вклад $\Gamma_m$, который дальше «переносится» по траектории
произведением матриц перехода — в точности как в непрерывной теории, где
$J_\theta(t) = \int \Phi(t, s) f_\theta(s)\, ds$ с фундаментальной матрицей $\Phi$.
$\Psi_e$ — дискретный аналог $\Phi(s_e, s_{e-1})$ (решения $\dot J_c = f_x J_c$ за элемент),
$\Gamma_e$ — дискретный аналог интеграла от $f_\theta$ за элемент.

### Стоимость и точность

**Стоимость.** LU-разложение $M$ уже сделано на последней итерации Ньютона. Для рекурсий
к нему добавляются только треугольные подстановки:

- $M^{-1} A$ — $n_x$ правых частей (столбцы $A$);
- $M^{-1} h_e B F_\theta$ — $n_\theta$ правых частей.

Итого $n_x + n_\theta$ подстановок с готовым LU размера $K n_x$ — чувствительности почти
бесплатны относительно самой симуляции. Сравните с текущей схемой: расширенная система
для `odeint` имеет размерность $n_x (1 + n_\theta + n_x)$ (для латеральной модели
$2 \cdot 8 = 16$ вместо $2$), и адаптивный солвер управляет шагом по всем компонентам сразу.

**Точность (internal numerical differentiation, Бок).** Полученные $S^\theta, S^c$ — это
**точные производные дискретной траектории**: продифференцирована сама численная схема,
а не непрерывная задача. Следствия:

1. якобиан согласован с невязками до машинной точности при любом $h_e$ — Гаусс–Ньютон
   получает точный градиент своей дискретной задачи и не «шумит» вблизи оптимума;
2. при интегрировании вариационных уравнений отдельным адаптивным солвером такой
   согласованности нет: у солвера свой допуск, и рассогласование якобиана с невязками
   порядка допуска ограничивает достижимую точность шага;
3. проверка (код-ячейка ниже): совпадение с конечными разностями той же схемы ~$10^{-9}$
   (предел точности разностей), с вариационными уравнениями ($\text{rtol}=10^{-12}$) — ~$10^{-12}$.


### Шаг 8. Строки измерений и накопление системы Гаусса–Ньютона

Пусть измерение $y_i$ приходится на конец элемента $e_i$ (сетка элементов привязана к сетке
измерений). Невязка и строка якобиана по вектору неизвестных $p = [\theta;\, c]$:

$$
r_i = y_i - h(x_{e_i}, \theta),
\qquad
J_i = \Bigl[\; \underbrace{-\bigl( h_x\, S^\theta_{e_i} + h_\theta \bigr)}_{n_y \times n_\theta}
\;\Big|\;
\underbrace{- h_x\, S^c_{e_i}}_{n_y \times n_x} \;\Bigr],
$$

где $h_x, h_\theta$ вычислены в $(x_{e_i}, \theta)$. Если измерение попадает **внутрь**
элемента (локальное время $\tau^{(i)}$), вместо последнего блока стадий берётся
интерполяция — чувствительности всех стадий $d z_e / d\theta$, $d z_e / d c$ уже вычислены
на шаге 6, достаточно заменить экстрактор $e_K^T$ на вектор значений базиса
$[\ell_1(\tau^{(i)}), \dots, \ell_K(\tau^{(i)})]$ (плюс вклад $\ell_0$ через $x_{e-1}$).

**Накопление.** Шагу Гаусса–Ньютона не нужна сама матрица $J$ размера $n_y N \times n_p$ —
только квадратная матрица нормальных уравнений и градиент:

$$
H_{\text{acc}} = \sum_{i=1}^{N} J_i^T W_i J_i \in \mathbb{R}^{n_p \times n_p},
\qquad
g_{\text{acc}} = \sum_{i=1}^{N} J_i^T W_i\, r_i \in \mathbb{R}^{n_p}.
$$

Обе суммы обновляются **по ходу марша по элементам**: прошли элемент → обновили
$S^\theta, S^c$ рекурсией → добавили вклад очередного измерения → двинулись дальше.
Память: $O(n_p^2)$ вместо $O(N n_y n_p)$; для single shooting $n_p = n_\theta + n_x$ —
крошечная плотная матрица. Шаг Гаусса–Ньютона:

$$
\bigl( H_{\text{acc}} + \text{reg} \bigr)\, \delta p = g_{\text{acc}} .
$$

Про гессиан: в Гауссе–Ньютоне «гессиан» — это и есть $J^T W J = H_{\text{acc}}$
(отброшен член $\sum_i r_i^T W_i \nabla^2 r_i$ с вторыми производными). Точный гессиан
потребовал бы вторых чувствительностей $\partial^2 x / \partial p^2$ — для ГН они не нужны,
и накапливать больше нечего.

### Шаг 9. Встраивание в multiple shooting

В multiple shooting неизвестные — $p = [\theta;\, c_1; \dots; c_T]$. Внутри шута $j$ рекурсии
запускаются заново с $S^c = I$, $S^\theta = 0$ (начало шута — его собственная переменная
$c_j$), и марш по элементам шута даёт:

- строки измерений шута: блоки $J_i$ попадают в столбцы $\theta$ и $c_j$
  (та же блочная структура $H$, что в `theory_gauss_newton.ipynb`);
- в конце шута — накопленные $S^\theta_{end} = J^j_\theta(\tau_j)$ и $S^c_{end} = J_{c_j}(\tau_j)$:
  ровно те блоки, из которых сейчас собирается матрица ограничений непрерывности $G$
  (в коде — `Jx_prev`, `Jc_prev` в `_solve_batch`).

То есть **меняется только интегратор**: вместо `get_jacobian_solution_*` (адаптивный
`odeint` расширенной системы) — марш по элементам с Ньютоном и рекурсиями $\Psi_e / \Gamma_e$.
Вся остальная машинерия — сборка $G$, $\mu$-регуляризованная ККТ (`gn_step`),
`run_optimization_adaptive`, ковариация через Шур-комплемент — работает без изменений.

### Сравнение двух подходов

| | Полная дискретизация (OCFE-NLP) | Рекурсивный интегратор (IRK + IND) |
|---|---|---|
| Неизвестные | $\theta$ + все узловые состояния | $\theta$ + $c_j$ — **как сейчас** |
| Матрицы | большая разреженная «стрелка» | малая плотная $H_{\text{acc}}$ |
| Хранение $J$ | разреженная, $O(N_e K)$ строк | не хранится (бегущие суммы) |
| Структура вычислений | все узлы сразу (батч/параллельно) | последовательный марш по элементам |
| Жёсткие системы | да (Радо IIA) | да (Радо IIA) |
| Бассейн сходимости | шире (lifting) | как у shooting |
| Пробный шаг | алгебраическая подстановка | полная симуляция (Ньютоны) |
| Вмешательство в код | новый класс `CollocationProblem` | замена интегратора (`CollocationIntegrator` вместо `VariationalIntegrator`, модель общая) |

Практический выбор: рекурсивный вариант — **минимальное вмешательство** (drop-in замена
`odeint`, сразу даёт жёсткие системы и якобианы, точно согласованные с невязками);
полная дискретизация — максимальный выигрыш по структуре (параллельная сборка, дешёвые
пробные шаги, лучший бассейн сходимости). Оба строятся на одних и тех же $\tilde D$,
узлах Радо и порядке $2K-1$.


In [1]:
import numpy as np

# ============================================================
# Узлы Радо IIA (K=3) и матрица дифференцирования
# ============================================================
tau_col = np.array([(4 - np.sqrt(6)) / 10,   # 0.155051...
                    (4 + np.sqrt(6)) / 10,   # 0.644949...
                    1.0])
K = len(tau_col)
nodes = np.concatenate([[0.0], tau_col])     # интерполяционные точки: tau_0=0 + узлы коллокации


def lagrange_basis(nodes, x):
    '''Значения базисных полиномов Лагранжа l_j(x), j = 0..K.'''
    L = np.ones(len(nodes))
    for j in range(len(nodes)):
        for m in range(len(nodes)):
            if m != j:
                L[j] *= (x - nodes[m]) / (nodes[j] - nodes[m])
    return L


def differentiation_matrix(nodes, at):
    '''D[k, j] = dl_j/dtau в точках at[k]; размер len(at) x len(nodes).'''
    n = len(nodes)
    D = np.zeros((len(at), n))
    for k, x in enumerate(at):
        for j in range(n):
            s = 0.0
            for m in range(n):
                if m == j:
                    continue
                prod = 1.0 / (nodes[j] - nodes[m])
                for r in range(n):
                    if r != j and r != m:
                        prod *= (x - nodes[r]) / (nodes[j] - nodes[r])
                s += prod
            D[k, j] = s
    return D


D = differentiation_matrix(nodes, tau_col)   # K x (K+1)
print('Узлы коллокации:', tau_col)
print('Ранг квадратного блока D[:, 1:]:', np.linalg.matrix_rank(D[:, 1:]), 'из', K)

# ============================================================
# Проверка порядка сходимости на dx/dt = lam*x, x(0) = 1
# (решаем поэлементно: (D[:,1:] - h*lam*I) X = -D[:,0]*x0)
# ============================================================
lam = -2.0
print('\nПорядок сходимости (ожидается 2K-1 = 5):')
prev = None
for N_e in (2, 4, 8, 16):
    h = 1.0 / N_e
    x0 = 1.0
    for e in range(N_e):
        A = D[:, 1:] - h * lam * np.eye(K)
        X = np.linalg.solve(A, -D[:, 0] * x0)
        x0 = X[-1]                            # tau_K = 1 — конец элемента
    err = abs(x0 - np.exp(lam))
    order = f'  order ~ {np.log2(prev / err):.2f}' if prev else ''
    print(f'  N_e = {N_e:3d}   err = {err:.3e}{order}')
    prev = err

# ============================================================
# L-устойчивость: жёсткий случай lam = -1e6, ОДИН элемент на [0, 1]
# (явный метод потребовал бы ~1e6 шагов; здесь достаточно одного)
# ============================================================
lam = -1e6
A = D[:, 1:] - 1.0 * lam * np.eye(K)
X = np.linalg.solve(A, -D[:, 0] * 1.0)
print(f'\nЖёсткий тест lam = -1e6, один элемент: x(1) = {X[-1]:.3e} (точное решение ~ 0)')


Узлы коллокации: [0.15505103 0.64494897 1.        ]
Ранг квадратного блока D[:, 1:]: 3 из 3

Порядок сходимости (ожидается 2K-1 = 5):
  N_e =   2   err = 3.318e-05
  N_e =   4   err = 1.091e-06  order ~ 4.93
  N_e =   8   err = 3.527e-08  order ~ 4.95
  N_e =  16   err = 1.124e-09  order ~ 4.97

Жёсткий тест lam = -1e6, один элемент: x(1) = 3.000e-06 (точное решение ~ 0)


In [2]:
import numpy as np
from scipy.integrate import solve_ivp

# ============================================================
# Верификация рекурсий Psi/Gamma на Лотке-Вольтерре (K = 3)
# ============================================================
d0, D1 = D[:, 0], D[:, 1:]                    # D построена выше
a_butcher = np.linalg.inv(D1)
print('Проверка A = 1⊗I  (то есть -D1^{-1} d0 = 1):',
      np.allclose(-a_butcher @ d0, np.ones(K)))
print('Таблица Бутчера a = D1^{-1} (Радо IIA, K=3):')
print(np.round(a_butcher, 6))

nx, nth = 2, 4
def f_lv(x, th):
    a, b, c, d = th
    return np.array([a*x[0] - b*x[0]*x[1], -c*x[1] + d*x[0]*x[1]])
def fx_lv(x, th):
    a, b, c, d = th
    return np.array([[a - b*x[1], -b*x[0]], [d*x[1], -c + d*x[0]]])
def fth_lv(x, th):
    return np.array([[x[0], -x[0]*x[1], 0, 0], [0, 0, -x[1], x[0]*x[1]]])

theta = np.array([1.2, 0.4, 0.3, 0.1])
c0 = np.array([6.0, 5.0])
T_end, N_e = 4.0, 200
h = T_end / N_e
E = np.zeros((nx, K*nx)); E[:, -nx:] = np.eye(nx)      # извлечение x_e = X_K
A_mat = np.kron(np.ones((K, 1)), np.eye(nx))
B_mat = np.kron(a_butcher, np.eye(nx))

def simulate_with_sens(c0, th):
    x, Sc, Sth = c0.copy(), np.eye(nx), np.zeros((nx, nth))
    for e in range(N_e):
        z = A_mat @ x                                   # старт Ньютона
        for _ in range(50):
            Xs = z.reshape(K, nx)
            F = np.concatenate([f_lv(Xs[k], th) for k in range(K)])
            JF = np.zeros((K*nx, K*nx))
            for k in range(K):
                JF[k*nx:(k+1)*nx, k*nx:(k+1)*nx] = fx_lv(Xs[k], th)
            M = np.eye(K*nx) - h * B_mat @ JF
            dz = np.linalg.solve(M, -(z - A_mat @ x - h * B_mat @ F))
            z += dz
            if np.max(np.abs(dz)) < 1e-14:
                break
        # рекурсии чувствительностей: то же M, новые правые части
        Xs = z.reshape(K, nx)
        JF = np.zeros((K*nx, K*nx)); Fth = np.zeros((K*nx, nth))
        for k in range(K):
            JF[k*nx:(k+1)*nx, k*nx:(k+1)*nx] = fx_lv(Xs[k], th)
            Fth[k*nx:(k+1)*nx] = fth_lv(Xs[k], th)
        M = np.eye(K*nx) - h * B_mat @ JF
        Psi   = E @ np.linalg.solve(M, A_mat)
        Gamma = E @ np.linalg.solve(M, h * B_mat @ Fth)
        Sc, Sth = Psi @ Sc, Psi @ Sth + Gamma
        x = z[-nx:]
    return x, Sc, Sth

xT, Sc, Sth = simulate_with_sens(c0, theta)

# 1) Сверка с конечными разностями ТОЙ ЖЕ дискретной схемы (точность IND)
eps = 1e-6
Sc_fd = np.column_stack([
    (simulate_with_sens(c0 + eps*np.eye(nx)[i], theta)[0]
     - simulate_with_sens(c0 - eps*np.eye(nx)[i], theta)[0]) / (2*eps)
    for i in range(nx)])
Sth_fd = np.column_stack([
    (simulate_with_sens(c0, theta + eps*np.eye(nth)[i])[0]
     - simulate_with_sens(c0, theta - eps*np.eye(nth)[i])[0]) / (2*eps)
    for i in range(nth)])
print(f'\nmax|S_c - конечные разности|     = {np.abs(Sc - Sc_fd).max():.2e}')
print(f'max|S_theta - конечные разности| = {np.abs(Sth - Sth_fd).max():.2e}')

# 2) Сверка с вариационными уравнениями (непрерывная «истина», rtol=1e-12)
def aug(t, y):
    x = y[:nx]
    Jt = y[nx:nx + nx*nth].reshape(nx, nth)
    Jc = y[nx + nx*nth:].reshape(nx, nx)
    return np.concatenate([f_lv(x, theta),
                           (fx_lv(x, theta) @ Jt + fth_lv(x, theta)).ravel(),
                           (fx_lv(x, theta) @ Jc).ravel()])
y0 = np.concatenate([c0, np.zeros(nx*nth), np.eye(nx).ravel()])
ref = solve_ivp(aug, (0, T_end), y0, rtol=1e-12, atol=1e-12).y[:, -1]
Jt_ref = ref[nx:nx + nx*nth].reshape(nx, nth)
Jc_ref = ref[nx + nx*nth:].reshape(nx, nx)
print(f'\nmax|S_theta - вариационные ур-я| = {np.abs(Sth - Jt_ref).max():.2e}')
print(f'max|S_c - вариационные ур-я|     = {np.abs(Sc - Jc_ref).max():.2e}')


Проверка A = 1⊗I  (то есть -D1^{-1} d0 = 1): True
Таблица Бутчера a = D1^{-1} (Радо IIA, K=3):
[[ 0.196815 -0.065535  0.023771]
 [ 0.394424  0.292073 -0.041549]
 [ 0.376403  0.512486  0.111111]]



max|S_c - конечные разности|     = 8.45e-10
max|S_theta - конечные разности| = 2.21e-09

max|S_theta - вариационные ур-я| = 5.94e-12
max|S_c - вариационные ур-я|     = 2.45e-13


## Накопление $H$ и $g$ по ходу марша — сверка с документом «MS and Orthogonal Collocations» и реализация

Внешний документ (sbauto.yonote.ru) описывает ту же конструкцию, что шаги 1–8 выше,
и доводит её до накопления градиента и гессиана. Сопоставление обозначений
(документ / этот ноутбук и код):

| документ | здесь / в коде | где проверено |
|---|---|---|
| $\mathbf{u} = (\mathbf{1}\otimes I)\,u_0 + h\,(D\otimes I)\,F(\mathbf{u})$ | $z_e = A\,x_{e-1} + h_e B\,F(z_e)$, $A = \mathbf{1}_K\otimes I$, $B = a\otimes I$ | шаги 3–4 |
| их $D$ (таблицы для $s=2,3$) | таблица Бутчера $a = D_1^{-1}$ (`RadauTables.butcher_a`) | `test_radau_tables` |
| их $d$ и тождество $D d = -\mathbf{1}$ | наш $d_0$ и тождество $-a\,d_0 = \mathbf{1}$ | ячейка выше |
| $C\,(I - hBF_z)^{-1}A$ | $\Psi_e = (e_K^\top\otimes I)\,M^{-1}A$ | `test_ind_property` |
| $C\,(I - hBF_z)^{-1}hBF_\theta$ | $\Gamma_e = (e_K^\top\otimes I)\,M^{-1}h_e B F_\theta$ | там же |
| $J^{(s)}_{k+1} = J^{(s)}_{k+1\mid k} J^{(s)}_k$, $\;J^{(\theta)}_{k+1} = J^{(s)}_{k+1\mid k} J^{(\theta)}_k + J^{(\theta)}_{k+1\mid k}$ | $S^c \leftarrow \Psi S^c$, $\;S^\theta \leftarrow \Psi S^\theta + \Gamma$ | шаг 7 |
| $g_{k+1} = g_k + J_{k+1}^\top r_{k+1}$; $H$ блоками $H^{(\theta)}, H^{(\theta s)}, H^{(s)}$ | `normal_equations.py::AccumulateMixin` — те же имена `H_theta`, `H_theta_c`, `H_c` | `test_accumulated_matches_dense` |

То есть интегратор и рекурсии якобианов из документа — в точности то, что уже
реализовано в `commom_utils/collocation.py`. **Новая часть документа — накопление:**
большая матрица $J \in \mathbb{R}^{Nm\times(p+n)}$ не материализуется вовсе, а
$g = \sum_i J_i^\top r_i$ и $H = \sum_i J_i^\top J_i$ собираются бегущими суммами
rank-$m$ обновлений по измерениям. Раньше код так не делал: `_solve_batch` стекал
строки в разреженную $J$, а шаг умножал $J^\top J$.

Мелкие опечатки в документе (по существу вся математика верна):
узлы $\tau$ записаны со знаменателем 16 — должно быть 10, $(4\pm\sqrt6)/10$;
размер $B = D\otimes I_n$ — $sn\times sn$, а не $sn\times n$.
Знаки согласованы с кодом: $r_i = y_i - h$, $g = J^\top r$ — правая часть
нормальных уравнений (антиградиент), как в `gn_step`.


### Обобщение накопления на multiple shooting (как реализовано)

Документ описывает single shooting: $p = [\theta;\, s]$, плотная
$H \in \mathbb{R}^{(n_\theta + n_x)\times(n_\theta + n_x)}$. В multiple shooting
$p = [\theta;\, c_1 \dots c_T]$, и та же сумма rank-$m$ обновлений даёт
**стрелочную** структуру: измерения шута $j$ затрагивают только столбцы
$[\theta;\, c_j]$, поэтому (в нотации документа $s$ — начальное состояние,
у нас это $c_j$ шута $j$)

$$
H = \begin{bmatrix}
H^{(\theta)} & H^{(\theta s)}_1 & \cdots & H^{(\theta s)}_T\\
(H^{(\theta s)}_1)^\top & H^{(s)}_1 & & \\
\vdots & & \ddots & \\
(H^{(\theta s)}_T)^\top & & & H^{(s)}_T
\end{bmatrix},
\qquad
g = \begin{bmatrix} g^{(\theta)} \\ g^{(s)}_1 \\ \vdots \\ g^{(s)}_T \end{bmatrix},
$$

$$
H^{(\theta)} = \sum_{j}\sum_{i \in j} (J^{(\theta)}_i)^\top J^{(\theta)}_i,
\qquad
H^{(\theta s)}_j = \sum_{i \in j} (J^{(\theta)}_i)^\top J^{(s)}_i,
\qquad
H^{(s)}_j = \sum_{i \in j} (J^{(s)}_i)^\top J^{(s)}_i,
$$

$$
g^{(\theta)} = \sum_{j}\sum_{i \in j} (J^{(\theta)}_i)^\top r_i,
\qquad
g^{(s)}_j = \sum_{i \in j} (J^{(s)}_i)^\top r_i,
$$

где $J^{(\theta)}_i = W_i\,(h_x S^\theta_i + h_\theta)$ и
$J^{(s)}_i = W_i\, h_x S^c_i$ — взвешенные блоки строки измерения $i$
(знак «минус» из шага 8 сокращается в $J^\top J$; в $g$ используется
соглашение кода $R = W(y - h)$ и правая часть $+J^\top R$, как в
`gn_step`). В коде это буквально `H_theta`, `H_theta_c`, `H_c`,
`g_theta`, `g_s`, а $J^{(\theta)}_i, J^{(s)}_i, r_i$ — поля `ShootRows`.

Существенные моменты:

- **Стыковки в $H$ не сворачиваются.** Строки $(J_G, R_G)$ нужны по отдельности:
  они стоят в седловой системе рядом с блоком $-\mu I$ (двойственные переменные)
  и в merit-функции $\Phi_\mu = \|R\|^2 + \frac1\mu\|R_G\|^2$. Их блоки — финальные
  $S^\theta, S^c$ каждого шута (`ShootRows.J_theta_end`, `J_s_end`).
- **Рекурсия = сумма.** Поточечные обновления $g_{k+1} = g_k + J_{k+1}^\top r_{k+1}$,
  $H_{k+1} = H_k + J_{k+1}^\top J_{k+1}$ из документа в коде векторизованы
  `einsum`-суммами по точкам шута — арифметически та же сумма.
- **Размеры.** Реальные данные (8000 точек, 10 шутов, 5 параметров): вместо
  $J$ размера $16000 \times 25$ и произведения $J^\top J$ — сразу $H$ размера
  $25 \times 25$ и $g \in \mathbb{R}^{25}$; sparse-сборка $J$ и умножение исчезают.

**Реализация — два слоя.** `gauss_newton/normal_equations.py` отвечает за то,
*как получить* $H$ и $g$: `NormalEquations.from_jacobian` (из построенной $J$)
или `AccumulateMixin.normal_equations` (накопление; классы
`MultipleShootingAccum`, `CollocationShootingAccum` — накопление не зависит от
интегратора). Общее для обоих путей ядро — метод `shoot_rows` в
`MultipleShooting`: интегрирование шутов, наблюдения и веса считаются один раз
и одинаково. `gauss_newton/adaptive.py` отвечает за то, *что с ними делать*:
шаг `gn_step` и цикл `run_optimization_adaptive` (Нильсен-$\lambda$ +
Пауэлл-$\mu$; стартовое $\mu = \|J_G\|_F^2 / \operatorname{tr}(H)$, поскольку
$\operatorname{tr}(H) = \|J\|_F^2$ — сама $J$ не нужна и здесь). Ковариация
$\theta$ тоже считается прямо из $H$: `NormalEquations.covariance_theta`.

Тесты: `pytests/accumulated_test.py` (совпадение с путём через $J$, ковариация,
идентификация), `pytests/collocation_accum_test.py` (сквозной прогон
`CollocationShootingAccum` с визуализацией). Верификация — код-ячейка ниже.


In [3]:
# ============================================================
# Верификация накопления: H, g из normal_equations() == J^T J, J^T R из solve()
# (Лотка-Вольтерра, 5 шутов; J строится только здесь, для сверки)
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from commom_utils.systems import LotkaVoltera
from commom_utils.ode_system import SyntheticDataGenerator
from gauss_newton.normal_equations import MultipleShootingAccum, NormalEquations
from gauss_newton.adaptive import gn_step

np.random.seed(0)
system = LotkaVoltera()
gen = SyntheticDataGenerator(system, sigma=0.01, perturb_initial=True,
                             perturbation_scale=0.0, use_jax=True)
t_b, meas_b, _, _ = gen.generate(c0=np.array([6.0, 5.0]),
                                 theta=np.array([1.2, 0.4, 0.3, 0.1]),
                                 time_intervals=[(0.0, 4.0)],
                                 n_measurements=50)
prob = MultipleShootingAccum(system=LotkaVoltera(), N_shoot=5,
                             gamma=np.ones(system.n_obs), c0_cost=1.0,
                             use_jax=True)
prob.add_batch(meas_b[0], t_b[0])
p0 = prob.make_full_theta(np.array([1.0, 0.5, 0.2, 0.05]))

J, R, J_G, R_G = prob.solve(p0)                    # путь через большую J
dense = NormalEquations.from_jacobian(J, R, J_G, R_G)
accum = prob.normal_equations(p0)                  # накопление без J

print('размер J:', J.shape, '->  размер H:', accum.H.shape)
print('|H - J^T J|_max         =', np.abs(accum.H.toarray() - dense.H.toarray()).max())
print('|g - J^T R|_max         =', np.abs(accum.g - dense.g).max())
print('|rss - ||R||^2|         =', abs(accum.rss - dense.rss))

d_dense, _ = gn_step(dense, mu=1.0, lam=1e-3)
d_accum, _ = gn_step(accum, mu=1.0, lam=1e-3)
print('|delta_dense - delta_accum| =', np.abs(d_dense - d_accum).max())


размер J: (98, 14) ->  размер H: (14, 14)
|H - J^T J|_max         = 4.547473508864641e-13
|g - J^T R|_max         = 3.552713678800501e-15
|rss - ||R||^2|         = 0.0
|delta_dense - delta_accum| = 8.204548151979907e-14


## План реализации

1. **Прототип на Лотке–Вольтерре** (`experiments/collocation/`) — два варианта на выбор:
   - **2a. Полная дискретизация:** класс `CollocationProblem` с интерфейсом
     `solve(theta_full) -> (J, R, J_G, R_G)` + аналог `make_full_theta` (узловые состояния
     из измерений через `inverse_h`); сборка $H$, $G$, $r$, $g$ по формулам выше.
     Максимальный выигрыш структуры: батчевая сборка, дешёвые пробные шаги, лучший бассейн
     сходимости.
   - **2b. Рекурсивный интегратор (IRK + IND):** марш по элементам с Ньютоном и рекурсиями
     $\Psi_e/\Gamma_e$ как замена `get_jacobian_solution_*` внутри `MultipleShooting`.
     Минимальное вмешательство: неизвестные и вся $\mu$-машинерия не меняются, сразу
     работают жёсткие системы.

   Оптимизация — существующим `run_optimization_adaptive` без изменений.
   **Критерий:** $\hat\theta$ и $\operatorname{Cov}(\hat\theta)$ совпадают с `MultipleShooting`
   в пределах доверительных интервалов.
2. **Векторизация:** батчевые `Function.map` для $f, f_x, f_\theta, h, h_x, h_\theta$ по всем узлам;
   сборка разреженных матриц из плотных блоков (по образцу `_solve_batch`); бенчмарк против
   multiple shooting; тест в `pytests/`.
3. **Реальные данные** (латеральная динамика): предвычисление $u$ в узлах, выбор $m$
   (элементов на интервал измерений), сравнение $\hat\theta$, доверительных интервалов и времени.
4. **Жёсткая система** (например, осциллятор Ван дер Поля с большим $\mu_{vdp}$):
   демонстрация случая, где текущий явный путь неприменим, а коллокации работают.
